# Synthetic Tabular Data Generation - Staged Optimization Driver

**Enhanced notebook for synthetic tabular data generation with staged hyperparameter optimization.**

This notebook provides a staged optimization workflow with:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: **Staged Hyperparameter Optimization** (NEW)
  - 4.1: Configuration
  - 4.2: Pilot phase (15 trials) with time estimates
  - 4.3: Continuation options
  - 4.4: Diminishing returns analysis
  - 4.5: Additional batches (optional)
- Section 5: Final models with best parameters

Based on STG-Driver-breast-cancer.ipynb with staged optimization enhancements.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print("="*60)
import sys                                                                                                                                                                                                     
!{sys.executable} -m pip install ctgan sdv ganeraid optuna -q

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[OK] Staged optimization module loaded from src/models/staged_optimization.py
[CONFIG] Session timestamp: 2026-02-19
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementa

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [2]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/Pakistani_Diabetes_Dataset.csv",  # Path to your CSV file
    "target_column": "Outcome",                 # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Pakistani Diabetes Dataset",      # Display name
    "dataset_identifier_override": None,          # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    "categorical_columns": ["Gender", "Rgn", "his", "dipsoa", "uria", "neph"],                    # List categorical cols, or [] for auto-detect
    "task_type": "auto",                          # "auto" | "classification" | "regression"

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,                      # True to sample rows for faster testing
    "sample_n": 500,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "none",                   # "none" | "drop" | "median" | "mode" | "mice" | "indicator_onehot"
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": "all",                       # "all" or list like ["ctgan", "tvae"]
    # "models_to_run": ["ctgan", "tvae", "copulagan"],  

    # ========== OPTIONAL: Tuning Configuration (for Section 3 demo) ==========
    "tuning_mode": "smoke",                       # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 5,                          # Trials for smoke testing
    "n_trials_full": 30,                          # Trials for full optimization
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/Pakistani_Diabetes_Dataset.csv
  Target column: Outcome
  Models to run: all
  Tuning mode: smoke


### 2.2 Load and Preprocess Data

In [3]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

# Load and preprocess data using the config
(
    data,                  # Processed DataFrame
    original_data,         # Copy for reference
    target_column,         # Target column name (cleaned)
    DATASET_IDENTIFIER,    # Dataset identifier for results paths
    categorical_columns,   # List of categorical columns
    metadata               # Full preprocessing metadata
) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

# Set aliases for backward compatibility with existing notebook cells
TARGET_COLUMN = target_column

# Get models to run based on config
models_to_run = get_models_to_run(NOTEBOOK_CONFIG)
tuning_config = get_tuning_config(NOTEBOOK_CONFIG)
N_TRIALS = get_n_trials(NOTEBOOK_CONFIG)

# Set RUN_MODE for backward compatibility with Section 4 tuning cells
RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  N_TRIALS: {N_TRIALS}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/Pakistani_Diabetes_Dataset.csv
[LOAD] Loaded 912 rows, 19 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (912, 19)
[PREPROCESS] Categorical columns: ['gender', 'rgn', 'his', 'dipsoa', 'uria', 'neph']
[PREPROCESS] Task type: classification
[PREPROCESS] Final shape: (912, 19)
[PREPROCESS] Dataset identifier: pakistani-diabetes-dataset

PREPROCESSING COMPLETE
  Dataset identifier: pakistani-diabetes-dataset
  Processed shape: (912, 19)
  Target column: outcome
  Task type: classification
  Categorical columns: ['gender', 'rgn', 'his', 'dipsoa', 'uria', 'neph']
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  RUN_MODE: debug
  N_TRIALS: 5
  Session: 2026-02-19
  Results path: results/pakistani-diabetes-dataset/2026-02-19/Section-2


### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [4]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Pakistani Diabetes Dataset
Target column: outcome
Results path: results/pakistani-diabetes-dataset/2026-02-19/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Pakistani Diabetes Dataset
   Shape......................... 912 rows x 19 columns
   Memory Usage.................. 0.13 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 2
   Numeric Columns............... 19
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (19 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.877 (Balanced)

[4/6] Feature Distributions...
   Saved: 3 distribution file(s) (18 features)

[5/6] Correlation Analysis...
   Saved: correlation_heatmap.png
   Saved: correlation_matrix.csv
   Saved: target_correlations.csv (18 features)

[6/6] Configuration Validati

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [5]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success':
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912


[1/7] Training CTGAN...
--------------------------------------------------
  Training CTGAN...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

  Generating 912 synthetic samples...
  [OK] CTGAN completed in 21.49s
  Synthetic data shape: (912, 19)

[2/7] Training TVAE...
--------------------------------------------------
  Training TVAE...
  Generating 912 synthetic samples...
[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 12.66s
  Synthetic data shape: (912, 19)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Training CopulaGAN...
  Generating 912 synthetic samples...
  [OK] CopulaGAN completed in 13.87s
  Synthetic data shape: (912, 19)

[4/7] Training CTABGAN...
--------------------------------------------------
  Training CTABGAN...


100%|██████████| 300/300 [00:33<00:00,  8.83it/s]


Finished training in 36.273441553115845  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN completed in 36.37s
  Synthetic data shape: (912, 19)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Training CTABGAN+...


100%|██████████| 400/400 [00:55<00:00,  7.22it/s]


Finished training in 57.662967681884766  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN+ completed in 57.75s
  Synthetic data shape: (912, 19)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Training PATE-GAN...
  Generating 912 synthetic samples...
  [OK] PATE-GAN completed in 10.64s
  Synthetic data shape: (912, 19)

[7/7] Training MEDGAN...
--------------------------------------------------
  Training MEDGAN...
  Generating 912 synthetic samples...
  [OK] MEDGAN completed in 14.63s
  Synthetic data shape: (912, 19)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 21.49s
  - TVAE: 12.66s
  - CopulaGAN: 13.87s
  - CTABGAN: 36.37s
  - CTABGAN+: 57.75s
  - PATE-GAN: 10.64s
  - MEDGAN: 14.63s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pategan', 'synthetic_data_medgan']


### 3.2 Batch Evaluation

In [6]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

section3_results = evaluate_all_available_models(
    section_number=3,
    scope=globals(),
    models_to_evaluate=None,
    real_data=None,
    target_col=None
)

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: pakistani-diabetes-dataset
[INFO] Target column: outcome
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/pakistani-diabetes-dataset/2026-02-19/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.811

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.034
   [CHART] Explained Variance (PC1, PC2): 0.303, 0.085

[3] DI

## 4 Staged Hyperparameter Optimization

This section uses a staged approach to hyperparameter optimization:
1. **Pilot phase**: Run a small number of trials to establish time estimates
2. **User decision**: Choose continuation strategy based on time estimates
3. **Continuation**: Run additional trials with chosen strategy
4. **Analysis**: Assess diminishing returns to decide when to stop

### 4.1 Configuration

Create the `StagedOptimizationManager` with configuration.

In [7]:
# Code Chunk ID: CHUNK_4_1_CONFIG
# ============================================================================
# SECTION 4.1 - STAGED OPTIMIZATION CONFIGURATION
# ============================================================================

from src.models.staged_optimization import (
    StagedOptimizationManager,
    StagedOptimizationConfig
)

# Configure staged optimization
staged_config = StagedOptimizationConfig(
    pilot_trials=10,           # Trials for pilot phase
    verbose_level=1,           # 0=silent, 1=concise, 2=detailed
    persistence_dir=f"results/{DATASET_IDENTIFIER}/optimization_studies",
    run_mode=RUN_MODE,         # "debug" or "full"
    random_state=42,
    continue_on_error=True
)

# Create the manager
manager = StagedOptimizationManager(
    data=data,
    target_column=target_column,
    categorical_columns=categorical_columns,
    dataset_identifier=DATASET_IDENTIFIER,
    config=staged_config
)

print("Staged Optimization Manager created!")
print(f"  Pilot trials: {staged_config.pilot_trials}")
print(f"  Run mode: {staged_config.run_mode}")
print(f"  Persistence dir: {staged_config.persistence_dir}")

Staged Optimization Manager created!
  Pilot trials: 10
  Run mode: debug
  Persistence dir: results/pakistani-diabetes-dataset/optimization_studies


### 4.2 Run Pilot Phase

Run a pilot phase with 15 trials per model to establish time estimates.
After this cell, you'll see:
- Average trial time per model
- Trials per hour estimates
- Projected time for additional trials

In [8]:
# Code Chunk ID: CHUNK_4_2_PILOT
# ============================================================================
# SECTION 4.2 - RUN PILOT PHASE (15 trials per model)
# ============================================================================

# Run pilot phase
pilot_states = manager.run_pilot(
    models_to_run=models_to_run,
    n_trials=5  # Pilot trials
)

# Display time estimates
print("\n" + "="*60)
print("PILOT PHASE COMPLETE - TIME ESTIMATES")
print("="*60)

time_estimates = manager.get_time_estimates()
if len(time_estimates) > 0:
    print(time_estimates.to_string(index=False))
else:
    print("No time estimates available")

# Show best scores so far
print("\nBest scores after pilot:")
for model_name, state in pilot_states.items():
    print(f"  {model_name}: {state.best_score:.4f} ({state.total_trials} trials)")

[I 2026-02-19 17:53:27,496] A new study created in memory with name: ctgan_hpo_pakistani-diabetes-dataset



STAGED OPTIMIZATION - PILOT PHASE
Models to optimize: 7
Pilot trials per model: 5
Dataset identifier: pakistani-diabetes-dataset


[PILOT] Optimizing CTGAN...
--------------------------------------------------


Gen. (-01.53) | Discrim. (+00.11): 100%|██████████| 250/250 [00:16<00:00, 15.03it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5679
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5609
[CHART] Combined Score: 0.5651 (Similarity: 0.5679, Accuracy: 0.5609)
[ctgan] Trial 1: Combined Score: 0.5651 (Similarity: 0.5679, Accuracy: 0.5609) Best Score so far: 0.5651


Gen. (-01.55) | Discrim. (+00.01): 100%|██████████| 200/200 [00:09<00:00, 20.03it/s]


...

Finished training in 78.18748450279236  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6027
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9649
[CHART] Combined Score: 0.7476 (Similarity: 0.6027, Accuracy: 0.9649)
[ctabganplus] Trial 3: Combined Score: 0.7476 (Similarity: 0.6027, Accuracy: 0.9649) Best Score so far: 0.7476


100%|██████████| 400/400 [01:37<00:00,  4.12it/s]


Finished training in 99.71082949638367  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5359
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9704
[CHART] Combined Score: 0.7097 (Similarity: 0.5359, Accuracy: 0.9704)
[ctabganplus] Trial 4: Combined Score: 0.7097 (Similarity: 0.5359, Accuracy: 0.9704) Best Score so far: 0.7476


100%|██████████| 400/400 [03:03<00:00,  2.19it/s]


Finished training in 186.1360776424408  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5640
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9781
[CHART] Combined Score: 0.7296 (Similarity: 0.5640, Accuracy: 0.9781)
[ctabganplus] Trial 5: Combined Score: 0.7296 (Similarity: 0.5640, Accuracy: 0.9781) Best Score so far: 0.7476
  [OK] CTABGAN+: 5 trials in 605.4s
  Best score: 0.7476

[PILOT] Optimizing PATE-GAN...
--------------------------------------------------
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3267
[OK] TRTS Evaluation: 2 scenarios, Average: 0.2456
[CHART] Combined Score: 0.2942 (Similarity: 0.3267, Accuracy: 0.2456)
[pategan] Trial 1: Combined Score: 0.2942 (Similarity: 0.3267, Accuracy: 0.2456) Best Score so far: 0.2942
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analy

In [9]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================
print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued               recommendation
      ctgan       5    0.581214          0.027759           0.016134          False Continue - insufficient data
       tvae       5    0.747212          0.127092           0.094964          False Continue - insufficient data
  copulagan       5    0.619567          0.266310           0.164997          False Continue - insufficient data
    ctabgan       5    0.762206          0.047687           0.036347          False Continue - insufficient data
ctabganplus       5    0.747579          0.024829           0.018561          False Continue - insufficient data
    pategan       5    0.369640          0.203972           0.075396          False Continue - insufficient data
     medgan       5    0.523231          0.373056           0.195195          False Continue - insufficient data

Interpretation:
----------------------------------------
  ctgan: 

### 4.3 Stage-2 Complement to Pilot

Based on the time estimates above, choose ONE of the following continuation strategies:

**Option (i): Common trials** - Same number of additional trials for all models
```python
manager.continue_optimization(additional_trials=30)
```

**Option (ii): Per-model trials** - Different trials per model
```python
manager.continue_optimization(trials_per_model={'ctgan': 50, 'tvae': 30, 'copulagan': 20})
```

**Option (iii): Time-based** - Specify time budget per model in minutes
```python
manager.continue_optimization(time_budget_minutes={'ctgan': 90, 'tvae': 60})
```

**Uncomment and modify ONE option below based on your needs:**

In [10]:
# Code Chunk ID: CHUNK_4_3_CONTINUE
# ============================================================================
# SECTION 4.3 - CONTINUATION (Choose ONE option)
# ============================================================================
# OPTION (i): Common trials for all models (commented out)
continuation_states = manager.continue_optimization(additional_trials=25)

# OPTION (ii): Per-model trials (commented out)
# continuation_states = manager.continue_optimization(
# trials_per_model={
# 'ctgan': 50,
# 'tvae': 30,
# 'copulagan': 20
# }
# )
# OPTION (iii): Time-based budget - 15 minutes per model
# Build time budget dict for all models being optimized
# time_budget = {model: 2 for model in models_to_run}
# print(f"Running optimization with 2 minutes per model:")
# for model, minutes in time_budget.items():
# print(f" {model}: {minutes} minutes")
# continuation_states = manager.continue_optimization(
# time_budget_minutes=time_budget
# )
# # Display results
# print("\n" + "="*60)
# print("CONTINUATION COMPLETE")
# print("="*60)
# print(manager.get_best_params_summary().to_string(index=False))


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 25 additional trials
  tvae: 25 additional trials
  copulagan: 25 additional trials
  ctabgan: 25 additional trials
  ctabganplus: 25 additional trials
  pategan: 25 additional trials
  medgan: 25 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 5 existing trials


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5502
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5839
[CHART] Combined Score: 0.5636 (Similarity: 0.5502, Accuracy: 0.5839)
[ctgan] Trial 6: Combined Score: 0.5636 (Similarity: 0.5502, Accuracy: 0.5839) Best Score so far: 0.5812


Gen. (-01.73) | Discrim. (-00.04): 100%|██████████| 250/250 [00:13<00:00, 18.91it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5786
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6014
[CHART] Combined Score: 0.5877 (Similarity: 0.5786, Accuracy: 0.6014)
[ctgan] Trial 7: Combined Score: 0.5877 (Similarity: 0.5786, Accuracy: 0.6014) Best Score so far: 0.5877


...

Finished training in 56.817286252975464  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5625
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9649
[CHART] Combined Score: 0.7234 (Similarity: 0.5625, Accuracy: 0.9649)
[ctabganplus] Trial 28: Combined Score: 0.7234 (Similarity: 0.5625, Accuracy: 0.9649) Best Score so far: 0.7519


100%|██████████| 300/300 [02:39<00:00,  1.88it/s]


Finished training in 162.7821204662323  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5774
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9803
[CHART] Combined Score: 0.7385 (Similarity: 0.5774, Accuracy: 0.9803)
[ctabganplus] Trial 29: Combined Score: 0.7385 (Similarity: 0.5774, Accuracy: 0.9803) Best Score so far: 0.7519


100%|██████████| 400/400 [01:24<00:00,  4.73it/s]


Finished training in 87.44232249259949  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5393
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9693
[CHART] Combined Score: 0.7113 (Similarity: 0.5393, Accuracy: 0.9693)
[ctabganplus] Trial 30: Combined Score: 0.7113 (Similarity: 0.5393, Accuracy: 0.9693) Best Score so far: 0.7519
  [OK] CTABGAN+: 25 trials in 1724.1s
  Best score: 0.7519

[CONTINUE] Optimizing PATE-GAN...
--------------------------------------------------
  Resuming from 5 existing trials
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3540
[OK] TRTS Evaluation: 2 scenarios, Average: 0.2336
[CHART] Combined Score: 0.3058 (Similarity: 0.3540, Accuracy: 0.2336)
[pategan] Trial 6: Combined Score: 0.3058 (Similarity: 0.3540, Accuracy: 0.2336) Best Score so far: 0.3696
[TARGET] Enhanced objective function using target

In [11]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================

print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued         recommendation
      ctgan      30    0.595401          0.006713           0.030321           True Stop - plateau reached
       tvae      30    0.764107          0.000000           0.111859           True Stop - plateau reached
  copulagan      30    0.725366          0.000000           0.270796           True Stop - plateau reached
    ctabgan      30    0.762206          0.000000           0.036347           True Stop - plateau reached
ctabganplus      30    0.751913          0.000000           0.022895           True Stop - plateau reached
    pategan      30    0.370153          0.000000           0.075910           True Stop - plateau reached
     medgan      30    0.523231          0.000000           0.195195           True Stop - plateau reached

Interpretation:
----------------------------------------
  ctgan: Stop - plateau reached
    -> Model has plateaue

### 4.5 Additional Batches (Optional)

If the diminishing returns analysis suggests continuing, run additional batches.
You can re-run this cell multiple times.

In [ ]:
# Code Chunk ID: CHUNK_4_5_ADDITIONAL
# ============================================================================
# SECTION 4.5 - ADDITIONAL BATCHES (Optional - can repeat)
# ============================================================================

# Skip this cell if you're satisfied with the current results
# Or uncomment and run to add more trials

# additional_states = manager.continue_optimization(additional_trials=20)
# 
# print("\nUpdated convergence report:")
# print(manager.get_diminishing_returns_report().to_string(index=False))

print("Cell skipped - uncomment to run additional batches")

### 4.6 Save Best Parameters

In [12]:
# Code Chunk ID: CHUNK_4_6_SAVE
# ============================================================================
# SECTION 4.6 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Extract studies from manager to globals for saving
for model_name, state in manager.model_states.items():
    if state.study is not None:
        globals()[f"{model_name}_study"] = state.study

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

[SAVE] SAVING BEST PARAMETERS FROM SECTION 4
[FOLDER] Target directory: results/pakistani-diabetes-dataset/2026-02-19/Section-4

[CHART] Processing CTGAN parameters...
[OK] Found CTGAN: 10 parameters, score: 0.5954

[CHART] Processing CTAB-GAN parameters...
[OK] Found CTAB-GAN: 2 parameters, score: 0.7622

[CHART] Processing CTAB-GAN+ parameters...
[OK] Found CTAB-GAN+: 2 parameters, score: 0.7519

[CHART] Processing GANerAid parameters...
[WARNING]  GANerAid: Study variable 'ganeraid_study' not found

[CHART] Processing CopulaGAN parameters...
[OK] Found CopulaGAN: 6 parameters, score: 0.7254

[CHART] Processing TVAE parameters...
[OK] Found TVAE: 8 parameters, score: 0.7641

[OK] Parameters saved: results/pakistani-diabetes-dataset/2026-02-19/Section-4/best_parameters.csv
   - Total parameter entries: 34
[OK] Summary saved: results/pakistani-diabetes-dataset/2026-02-19/Section-4/best_parameters_summary.csv
   - Models processed: 5

[SAVE] Parameter saving completed!
[FOLDER] Files sa

## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [13]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),
    verbose=True
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")


SECTION 5.1: BATCH TRAINING WITH BEST PARAMETERS
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912
Loading parameters from: Section 4

[1/3] Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/pakistani-diabetes-dataset/2026-02-19/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 5
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - copulagan: 6 parameters
   - tvae: 8 parameters
   Parameter source: CSV file
   Models with parameters: 5

[2/3] Training models with optimized parameters...

[1/7] Training CTGAN...
--------------------------------------------------
  Parameters load

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml fo

  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5400
[OK] TRTS Evaluation: 2 scenarios, Average: 0.4934
[CHART] Combined Score: 0.5213 (Similarity: 0.5400, Accuracy: 0.4934)
  [OK] CTGAN completed in 33.83s
  Synthetic data shape: (912, 19)
  Objective score: 0.5213

[2/7] Training TVAE...
--------------------------------------------------
  Parameters loaded: 8
    - epochs: 300
    - batch_size: 128
    - learning_rate: 0.0049
    - embedding_dim: 64
    - l2scale: 0.0005
    ... and 4 more
  Training TVAE...
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6066
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9463
[CHART] Combined Score: 0.7425 (Similarity: 0.6066, Accuracy: 0.9463)
  [OK] TVAE completed in 21.83s
  Synthetic data shape: (912, 19)
  Objective

100%|██████████| 400/400 [01:10<00:00,  5.69it/s]


Finished training in 74.68342995643616  seconds.
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5474
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9545
[CHART] Combined Score: 0.7103 (Similarity: 0.5474, Accuracy: 0.9545)
  [OK] CTABGAN completed in 74.81s
  Synthetic data shape: (912, 19)
  Objective score: 0.7103

[5/7] Training CTABGAN+...
--------------------------------------------------
  Parameters loaded: 2
    - epochs: 350
    - batch_size: 512
    - categorical_columns: ['gender', 'rgn', 'his', 'uria', 'neph']
    - target_col: outcome
  Training CTABGAN+...


100%|██████████| 350/350 [01:13<00:00,  4.75it/s]


Finished training in 76.9413743019104  seconds.
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5675
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9781
[CHART] Combined Score: 0.7317 (Similarity: 0.5675, Accuracy: 0.9781)
  [OK] CTABGAN+ completed in 77.20s
  Synthetic data shape: (912, 19)
  Objective score: 0.7317

[6/7] Training PATE-GAN...
--------------------------------------------------
  Parameters loaded: 0
    - discrete_columns: ['gender', 'rgn', 'his', 'uria', 'neph']
  Training PATE-GAN...
  Generating 912 synthetic samples...
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.4127
[ERROR] TRTS (Real->Synthetic) failed: Classification metrics can't handle a mix of continuous and binary targets
[ERROR] TRTS (Synthetic->Real) failed: Unknown label type: continuous. Maybe you are trying t

### 5.2 Batch Evaluation of Optimized Models

In [14]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

try:
    section5_batch_results = evaluate_section5_optimized_models(
        section_number=5,
        scope=globals(),
        target_column=TARGET_COLUMN
    )
    
    print("\n" + "="*80)
    print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
    print("="*80)
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
    
except Exception as e:
    print(f"Section 5.2 batch evaluation failed: {e}")
    import traceback
    traceback.print_exc()

SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 5
[INFO] Dataset: pakistani-diabetes-dataset
[INFO] Target column: outcome
[INFO] Variable pattern: final
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/pakistani-diabetes-dataset/2026-02-19/Section-5/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.813

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.028
   [CHART] Explained Variance (PC1, PC2): 0.303, 0.085

[3] DISTRIBUTION SIMILARITY
------------------------------
   [CHART] Aver

### 5.3 Final Summary

In [15]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show staged optimization summary
if 'manager' in dir() and manager is not None:
    print(f"\nStaged Optimization Summary:")
    print("-" * 60)
    summary = manager.get_best_params_summary()
    if len(summary) > 0:
        print(summary[['model', 'best_score', 'total_trials', 'status']].to_string(index=False))

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)

SYNTHETIC DATA GENERATION - FINAL SUMMARY

Dataset: pakistani-diabetes-dataset
Session: 2026-02-19

Results directories:
  Section 2 (EDA): results/pakistani-diabetes-dataset/2026-02-19/Section-2
  Section 3 (Demo): results/pakistani-diabetes-dataset/2026-02-19/Section-3
  Section 4 (HPO): results/pakistani-diabetes-dataset/2026-02-19/Section-4
  Section 5 (Final): results/pakistani-diabetes-dataset/2026-02-19/Section-5

Staged Optimization Summary:
------------------------------------------------------------
      model  best_score  total_trials    status
      ctgan    0.595401            30 completed
       tvae    0.764107            30 completed
  copulagan    0.725366            30 completed
    ctabgan    0.762206            30 completed
ctabganplus    0.751913            30 completed
    pategan    0.370153            30 completed
     medgan    0.523231            30 completed

Final Model Performance (with optimized parameters):
-----------------------------------------------